# Phase 3: Student Distillation Training (Colab / Gemini Ultra)

Self-contained notebook to pull all scored Parquet shards from `christianengel/rerank-mini-distill-shards` (`track_a_prescored/` + `track_b_scored/`), run Stage 1 (512-tok) and Stage 2 (4K-tok long-context curriculum) distillation onto `jhu-clsp/ettin-encoder-32m`, and push the resulting checkpoint to a private Hugging Face staging model repo.

In [ ]:
!pip -q install -U 'transformers>=4.51' 'sentence-transformers>=5.0' 'huggingface_hub>=0.30' pyarrow tqdm

In [ ]:
import os
from pathlib import Path
from huggingface_hub import HfApi, snapshot_download

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN') or os.environ.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')
assert HF_TOKEN, 'Set HF_TOKEN in Colab Secrets or os.environ'
os.environ['HF_TOKEN'] = HF_TOKEN

SHARDS_REPO = 'christianengel/rerank-mini-distill-shards'
OUT_MODEL_REPO = 'christianengel/rerank-mini-32m-staging'
LOCAL_DATA = Path('/content/distill_shards')
snapshot_download(
    repo_id=SHARDS_REPO, repo_type='dataset', token=HF_TOKEN,
    local_dir=str(LOCAL_DATA), allow_patterns=['track_a_prescored/*.parquet', 'track_b_scored/*.parquet']
)
print('Downloaded shards:', len(list(LOCAL_DATA.rglob('*.parquet'))))

In [ ]:
# Clone the training script from GitHub or upload train.py into /content/
# Stage 1: 512-token warmup across full Track A + Track B pool
!python train.py \
  --data-dirs /content/distill_shards/track_a_prescored /content/distill_shards/track_b_scored \
  --out-dir /content/ckpt_stage1 \
  --backbone jhu-clsp/ettin-encoder-32m --pooling cls \
  --loss hybrid --tau 1.0 --mse-weight 0.05 --margin-weight 0.05 \
  --docs-per-query 8 --max-length 512 --batch-size 16 --grad-accum 2 \
  --lr 8e-5 --weight-decay 0.01 --epochs 1.0 --code-upsample 2

# Stage 2: 4096-token long-context + code curriculum
!python train.py \
  --data-dirs /content/distill_shards/track_a_prescored /content/distill_shards/track_b_scored \
  --out-dir /content/ckpt_stage2 \
  --backbone /content/ckpt_stage1 --pooling cls \
  --loss hybrid --tau 1.0 --mse-weight 0.05 --margin-weight 0.05 \
  --docs-per-query 8 --max-length 4096 --batch-size 4 --grad-accum 8 \
  --lr 3e-5 --weight-decay 0.01 --max-steps 1500 --long-doc-upsample 6 --grad-checkpointing

In [ ]:
api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=OUT_MODEL_REPO, repo_type='model', private=True, exist_ok=True)
api.upload_folder(folder_path='/content/ckpt_stage2', repo_id=OUT_MODEL_REPO, repo_type='model')
print('Uploaded checkpoint to', OUT_MODEL_REPO)